# Phase 5: Feature Extraction
This notebook describes the feature engineering stage of the exoplanet pipeline.

### Target Goals:
Translate variable-length astronomical light curves into a fixed-length **12-dimensional feature vector** per star, suitable for training classical machine learning models (Random Forest, XGBoost).

### Feature Descriptions:
1. `bls_period`: Orbital period recovered by BLS.
2. `bls_depth`: Fractional transit depth.
3. `bls_duration`: Transit duration in days.
4. `bls_power`: Power significance of the BLS peak.
5. `transit_snr`: Signal-to-noise ratio ($Depth / OOT\_Scatter$).
6. `mean_to_max_depth_ratio`: **U-vs-V shape metric**. Confirmed planets (U-shape) have values $>0.75$, while eclipsing binaries (V-shape) have values around $\sim 0.5$.
7. `in_transit_skew`: Skewness of the folded flux within transit.
8. `in_transit_kurtosis`: Kurtosis of the folded flux within transit.
9. `std_raw`: Standard deviation of the raw light curve (evaluates overall stellar variability).
10. `std_detrended`: Out-of-transit scatter in the flattened curve.
11. `transit_count`: Expected number of transits spanned by the observation window.
12. `flux_kurtosis`: Overall kurtosis of the flattened light curve.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Setup complete.")

## 1. Feature Matrix Output
We load the extracted feature matrix from `outputs/features.csv`.

In [2]:
features_df = pd.read_csv('../outputs/features.csv')
features_df.head(10)

star_id,label,catalog_period,bls_period,bls_depth,bls_duration,bls_power,transit_snr,mean_to_max_depth_ratio,in_transit_skew,in_transit_kurtosis,std_raw,std_detrended,transit_count,flux_kurtosis
231663901,confirmed_planet,1.430370,1.429927,0.009182,0.037,3.502241e-04,2.224710,0.426121,0.004532,-0.101261,8.946216,0.004399,19.497516,0.598485
149603524,confirmed_planet,4.411937,4.412548,0.001570,0.143,2.139236e-06,1.714588,0.251029,0.123630,-0.792556,50.872373,0.001086,6.318498,5.655807
336732616,confirmed_planet,3.547854,3.549894,0.001350,0.117,5.764748e-06,0.592195,0.125265,-0.002389,0.017945,12.056742,0.002337,7.815069,0.131719
231670397,confirmed_planet,4.087296,4.089057,0.000769,0.037,1.669127e-07,0.823648,0.279710,0.226277,-0.179989,21.648539,0.000938,6.818204,0.197615
144065872,confirmed_planet,2.184667,2.185589,0.002019,0.090,3.937971e-06,2.262170,0.232267,-1.394069,4.282307,61.731274,0.001090,12.756584,4.832753
38846515,confirmed_planet,2.849383,2.849480,0.001544,0.037,1.231530e-06,1.272597,0.281892,0.132513,1.290542,22.905939,0.001228,9.160156,0.233798
92352620,confirmed_planet,3.950201,3.948171,0.002566,0.037,1.064390e-06,2.177969,0.412328,0.186583,-0.382227,59.540220,0.001217,7.027049,1.609737
289793076,confirmed_planet,3.044049,3.042662,0.004440,0.090,1.125854e-04,0.539924,0.153116,0.148537,0.189353,7.905485,0.008313,9.118403,0.057430
29344935,confirmed_planet,2.766760,2.768049,0.002853,0.090,4.693972e-05,0.372006,0.093686,0.050778,-0.183881,7.038452,0.007730,10.017450,0.028590
281459670,confirmed_planet,3.174350,3.176881,0.002384,0.090,1.018955e-05,0.422030,0.133051,-0.159074,-0.160267,26.799673,0.005680,8.776183,2.678056


## 2. Feature Analysis: U-vs-V Shape Metric
Let's check how the `mean_to_max_depth_ratio` separates planet transits (boxy, value $\approx 0.8$) from eclipsing binaries (pointed, value $\approx 0.5$).